# Property Graph Index (GraphRAG)

Every index so far retrieves by semantic similarity — great for "what is X," weaker for "how does X relate to Y." A `PropertyGraphIndex` has an LLM read each chunk and extract entity → relationship → entity triplets (e.g. "Naruto → is jinchuriki of → Kurama"), building an actual graph you can query for relationships directly instead of hoping the right chunk happens to mention both entities together.


**Step 1 — Setup.** Patch asyncio for notebook compatibility, quiet the logs, load API keys, and set the embedding model plus a separate LLM dedicated to entity/relationship extraction.


In [1]:
import logging

import nest_asyncio
from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# PropertyGraphIndex's extraction step calls asyncio.run() internally, which
# conflicts with the event loop Jupyter already runs — nest_asyncio patches
# around that so extraction can run inside a notebook cell.
nest_asyncio.apply()

# Quiet down noisy INFO-level logs from httpx, llama_index, and openai.
for noisy_logger in ("httpx", "llama_index", "openai"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Load API keys from .env into the environment.
load_dotenv()

# Only the embedding model is set globally here — the extraction LLM below is
# passed explicitly instead of going through Settings.llm.
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# gpt-4o-mini is used for extraction (not the project-wide gpt-4.1-nano default)
# because reliably identifying entities and relationships benefits from a more
# capable model, the same reasoning used for the router and agent episodes.
extraction_llm = OpenAI(model="gpt-4o-mini")

**Step 2 — Extract a graph from two documents.** `PropertyGraphIndex.from_documents()` runs `SimpleLLMPathExtractor` over each chunk, asking the LLM to pull out entity → relationship → entity triplets, then stores them all in an in-memory graph.


In [ ]:
from llama_index.core import PropertyGraphIndex, SimpleDirectoryReader
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor

# Using 2 of the 5 anime docs keeps extraction fast for this demo — the same
# pattern scales to the full corpus, just with more LLM extraction calls.
documents = SimpleDirectoryReader(
    input_files=[
        "data/sample_docs/naruto.txt",
        "data/sample_docs/demon_slayer.txt",
    ]
).load_data()

# Builds the graph: each chunk goes through kg_extractors, which asks the LLM
# to output (subject, relation, object) triplets that get added to the graph.
graph_index = PropertyGraphIndex.from_documents(
    documents,
    llm=extraction_llm,  # index-level default LLM, used by retrieval (e.g. the LLMSynonymRetriever), not by extraction
    kg_extractors=[
        SimpleLLMPathExtractor(llm=extraction_llm)
    ],  # this LLM is the one that actually performs triplet extraction
    show_progress=False,
)

# get_triplets() with no filters intentionally returns nothing — the underlying
# graph object holds everything that was extracted.
triplets = graph_index.property_graph_store.graph.get_triplets()
print(f"Extracted {len(triplets)} relationship triplets, e.g.:\n")
for subject, relation, obj in triplets[:8]:  # each triplet is (subject, relation, object)
    print(f"{subject.name} --[{relation.label}]--> {obj.name}")

Extracted 40 relationship triplets, e.g.:

Naruto --[Is voiced by]--> Maile flanagan
Naruto --[Has sold]--> Over 250 million copies
Naruto --[Is grouped with]--> One piece and bleach
Naruto --[Has total]--> 720 television episodes
Naruto --[Confronts]--> Pain
Naruto --[Learns]--> Sage mode
Demon slayer --[Serialized in]--> Weekly shonen jump
Tanjiro --[Fights using]--> Water breathing


**Step 3 — Ask a relationship question.** `as_query_engine()` on a property graph index still works like any other LlamaIndex query engine, but under the hood it retrieves relevant triplets from the graph instead of similar text chunks — perfect for a "how is X related to Y" question.


In [3]:
from llama_index.llms.openai import OpenAI

# `as_query_engine(llm=...)` only sets the LLM used for response synthesis (the
# final answer). Retrieval uses a separate LLMSynonymRetriever, built with
# whatever `llm` was passed to `from_documents()` above (extraction_llm,
# gpt-4o-mini) — so retrieval stays on gpt-4o-mini while a smaller gpt-4.1-nano
# only has to turn already-retrieved triplets into a sentence.
query_engine = graph_index.as_query_engine(llm=OpenAI(model="gpt-4.1-nano"))
response = query_engine.query("How is Kurama related to Naruto?")
print(response)

Naruto Uzumaki is the Jinchuriki of Kurama.


**Three LLMs, three separate jobs.** It's easy to mix these up since two of them point at the same `extraction_llm` object — here's where each one actually runs:

```
documents (text)
      │
      ▼
[1] extraction_llm inside SimpleLLMPathExtractor   (runs once, at index-build time)
      │  reads each chunk → outputs (subject, relation, object) triples
      ▼
triples saved in graph_index.property_graph_store
      │
      │        your question ("How is Kurama related to Naruto?")
      │              │
      │              ▼
      │        [2] extraction_llm as the index's default llm=,
      │            used by the retriever (LLMSynonymRetriever)   (runs on every .query())
      │              │  expands your question into candidate entity names/synonyms
      │              ▼
      └────────► matching triples found in the graph
                       │
                       ▼
                 [3] llm=OpenAI("gpt-4.1-nano") passed to as_query_engine()   (runs on every .query())
                       │  turns the retrieved triples into a natural-language answer
                       ▼
                 final response
```

- **[1] Extraction** — text → triples. One-time cost per document, done while building the index.
- **[2] Retrieval** — question → graph lookup. Runs every query; only rewrites/expands the question, never touches raw text.
- **[3] Synthesis** — triples → sentence. Runs every query; this is the only LLM whose output you actually read.


### Summary

- `SimpleLLMPathExtractor` reads each node and asks the LLM to output entity-relationship-entity triplets, which get stored in a graph store (`graph_index.property_graph_store`) rather than just a flat list of embedded chunks.
- Querying a property graph index still goes through a normal `.query()` call — under the hood it retrieves relevant triplets (and optionally nearby text) instead of just similar chunks, which is what makes relationship questions like "how is X related to Y" answerable directly.
- This is the current "GraphRAG" pattern — worth reaching for when your documents describe a genuine network of entities and relationships, not just independent facts.
